# Serverless cleaning validation — 5 geometry-cleaning functions

Proves the light-tier geometry-cleaning functions work end-to-end on real Serverless (e2-demo / oauth-fe).

**Five stages (one per function):**
1. `gbx_st_simplifypreservetopology` — near-collinear polygon vertex dropped, topology preserved, valid
2. `gbx_st_removerepeatedpoints` — consecutive duplicates removed from a line
3. `gbx_st_reduceprecision` — off-grid coordinates snapped to a 1.0 grid
4. `gbx_st_node` — self-intersecting linework noded at the crossing, valid
5. `gbx_st_snap` — near-miss line vertices aligned to reference within tolerance
6. Benchmark — throughput for each function (synthetic polygon corpus, repartition by id)

In [ ]:
import datetime
import json
import time

from shapely import from_wkb, is_valid

N_PART = 8
N_BENCH = 1000

results = {
    "run_date": datetime.datetime.utcnow().isoformat() + "Z",
    "functions_tested": 5,
    "stages": {},
}

print(f"Cleaning validation: 5 functions, benchmark {N_BENCH} rows x {N_PART} partitions")

In [ ]:
from databricks.labs.gbx.pyvx import functions as pyvx_functions

pyvx_functions.register(spark, only=[
    "gbx_st_simplifypreservetopology",
    "gbx_st_removerepeatedpoints",
    "gbx_st_reduceprecision",
    "gbx_st_node",
    "gbx_st_snap",
])
print("Registered: gbx_st_simplifypreservetopology, gbx_st_removerepeatedpoints, "
      "gbx_st_reduceprecision, gbx_st_node, gbx_st_snap")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 1: gbx_st_simplifypreservetopology
# Near-collinear polygon vertex dropped; topology preserved; output valid.
# ---------------------------------------------------------------------------
INPUT_WKT_SIMPLIFY = "POLYGON((0 0,0 5,0.001 8,0 10,10 10,10 0,0 0))"
TOLERANCE = 1.0
# The near-collinear vertex (0.001 8) should be dropped at tolerance=1.0.
# Original exterior ring has 7 coords (incl. closing coord).
ORIGINAL_EXTERIOR_COORDS = 7

df = spark.createDataFrame([(1, INPUT_WKT_SIMPLIFY)], "id INT, wkt STRING")
row = df.repartition(1, "id").selectExpr(
    f"gbx_st_simplifypreservetopology(wkt, {TOLERANCE}) AS simplified"
).collect()[0]

simplified_wkb = row["simplified"]
assert simplified_wkb is not None, "gbx_st_simplifypreservetopology returned NULL"

g = from_wkb(bytes(simplified_wkb))
n_coords = len(g.exterior.coords)
is_polygon = g.geom_type == "Polygon"
geom_valid = bool(is_valid(g))
fewer_vertices = n_coords < ORIGINAL_EXTERIOR_COORDS

print(f"simplifypreservetopology:")
print(f"  input exterior_coords={ORIGINAL_EXTERIOR_COORDS}, output exterior_coords={n_coords}")
print(f"  geom_type={g.geom_type}, valid={geom_valid}, fewer_vertices={fewer_vertices}")

assert geom_valid, f"Output not valid: {g.geom_type}"
assert is_polygon, f"Expected Polygon, got {g.geom_type} (topology not preserved)"
assert fewer_vertices, (
    f"Expected < {ORIGINAL_EXTERIOR_COORDS} exterior coords after simplify, got {n_coords}"
)

results["stages"]["simplifypreservetopology"] = {
    "pass": True,
    "effect": "fewer_vertices",
    "input_exterior_coords": ORIGINAL_EXTERIOR_COORDS,
    "output_exterior_coords": n_coords,
    "geom_type": g.geom_type,
    "valid": geom_valid,
}
print("PASS: simplifypreservetopology — topology preserved, fewer vertices")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 2: gbx_st_removerepeatedpoints
# Consecutive duplicate vertices removed from a line.
# ---------------------------------------------------------------------------
INPUT_WKT_DUP = "LINESTRING(0 0,0 0,1 1,1 1,2 2)"
EXPECTED_COORDS = [(0.0, 0.0), (1.0, 1.0), (2.0, 2.0)]
INPUT_COORD_COUNT = 5  # 5 coords in the input (two pairs of duplicates)

df = spark.createDataFrame([(1, INPUT_WKT_DUP)], "id INT, wkt STRING")
row = df.repartition(1, "id").selectExpr(
    "gbx_st_removerepeatedpoints(wkt) AS deduped"
).collect()[0]

deduped_wkb = row["deduped"]
assert deduped_wkb is not None, "gbx_st_removerepeatedpoints returned NULL"

g = from_wkb(bytes(deduped_wkb))
actual_coords = list(g.coords)
output_coord_count = len(actual_coords)
dupes_removed = output_coord_count < INPUT_COORD_COUNT
coords_match = actual_coords == EXPECTED_COORDS

print(f"removerepeatedpoints:")
print(f"  input_coords=5, output_coords={output_coord_count}")
print(f"  actual={actual_coords}, expected={EXPECTED_COORDS}")

assert dupes_removed, (
    f"Expected {INPUT_COORD_COUNT} -> {len(EXPECTED_COORDS)} coords, got {output_coord_count}"
)
assert coords_match, f"Coords mismatch: expected {EXPECTED_COORDS}, got {actual_coords}"

results["stages"]["removerepeatedpoints"] = {
    "pass": True,
    "effect": "dedup",
    "input_coord_count": INPUT_COORD_COUNT,
    "output_coord_count": output_coord_count,
    "coords_match_expected": coords_match,
}
print("PASS: removerepeatedpoints — consecutive duplicates removed")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 3: gbx_st_reduceprecision
# Off-grid coordinates snapped to a 1.0 grid (a.k.a. snap-to-grid).
# ---------------------------------------------------------------------------
INPUT_WKT_OFFGRID = "POINT(1.234 5.678)"
GRID_SIZE = 1.0
EXPECTED_COORD = (1.0, 6.0)  # 1.234 rounds to 1.0; 5.678 rounds to 6.0

df = spark.createDataFrame([(1, INPUT_WKT_OFFGRID)], "id INT, wkt STRING")
row = df.repartition(1, "id").selectExpr(
    f"gbx_st_reduceprecision(wkt, {GRID_SIZE}) AS gridded"
).collect()[0]

gridded_wkb = row["gridded"]
assert gridded_wkb is not None, "gbx_st_reduceprecision returned NULL"

g = from_wkb(bytes(gridded_wkb))
actual_coord = list(g.coords)[0]
coord_snapped = (
    abs(actual_coord[0] - EXPECTED_COORD[0]) < 1e-9
    and abs(actual_coord[1] - EXPECTED_COORD[1]) < 1e-9
)

print(f"reduceprecision:")
print(f"  input=(1.234, 5.678), grid_size={GRID_SIZE}")
print(f"  output={actual_coord}, expected={EXPECTED_COORD}")

assert coord_snapped, f"Expected coord {EXPECTED_COORD}, got {actual_coord}"

results["stages"]["reduceprecision"] = {
    "pass": True,
    "effect": "snapped_to_grid",
    "grid_size": GRID_SIZE,
    "input_coord": [1.234, 5.678],
    "output_coord": list(actual_coord),
    "expected_coord": list(EXPECTED_COORD),
}
print("PASS: reduceprecision — off-grid coords snapped to 1.0 grid")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 4: gbx_st_node
# Self-intersecting linework noded at the crossing — output is valid.
# ---------------------------------------------------------------------------
INPUT_WKT_XING = "LINESTRING(0 0,10 10,0 10,10 0)"  # figure-eight (self-intersects at 5 5)

df = spark.createDataFrame([(1, INPUT_WKT_XING)], "id INT, wkt STRING")
row = df.repartition(1, "id").selectExpr(
    "gbx_st_node(wkt) AS noded"
).collect()[0]

noded_wkb = row["noded"]
assert noded_wkb is not None, "gbx_st_node returned NULL"

g = from_wkb(bytes(noded_wkb))
geom_type = g.geom_type
geom_valid = bool(is_valid(g))
is_line_type = geom_type in ("MultiLineString", "LineString")
noded_parts = len(list(g.geoms)) if geom_type == "MultiLineString" else 1

print(f"node:")
print(f"  geom_type={geom_type}, valid={geom_valid}, noded_parts={noded_parts}")

assert geom_valid, f"Noded output not valid: {g!r}"
assert is_line_type, f"Expected MultiLineString or LineString, got {geom_type}"
# Noded output should have at least 1 part; MultiLineString confirms crossing was detected
if geom_type == "MultiLineString":
    assert noded_parts >= 2, f"Expected >= 2 noded parts, got {noded_parts}"

results["stages"]["node"] = {
    "pass": True,
    "effect": "noded_parts",
    "geom_type": geom_type,
    "valid": geom_valid,
    "noded_parts": noded_parts,
}
print(f"PASS: node — self-intersecting linework noded into {noded_parts} part(s), valid")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 5: gbx_st_snap
# Near-miss line (y=0.4) snaps onto reference (y=0) within tolerance 0.5.
# ---------------------------------------------------------------------------
INPUT_WKT_NEAR = "LINESTRING(0 0.4,10 0.4)"
REF_WKT = "LINESTRING(0 0,10 0)"
TOLERANCE_SNAP = 0.5

df = spark.createDataFrame(
    [(1, INPUT_WKT_NEAR, REF_WKT)],
    "id INT, geom STRING, ref STRING",
)
row = df.repartition(1, "id").selectExpr(
    f"gbx_st_snap(geom, ref, {TOLERANCE_SNAP}) AS snapped"
).collect()[0]

snapped_wkb = row["snapped"]
assert snapped_wkb is not None, "gbx_st_snap returned NULL"

g = from_wkb(bytes(snapped_wkb))
coords = list(g.coords)
y_snapped = all(abs(y) < 1e-9 for _, y in coords)

print(f"snap:")
print(f"  input y=0.4, tolerance={TOLERANCE_SNAP}, reference y=0")
print(f"  output coords={coords}")

assert y_snapped, (
    f"Expected all y≈0 (snapped to reference), got coords={coords}"
)

results["stages"]["snap"] = {
    "pass": True,
    "effect": "aligned_vertices",
    "tolerance": TOLERANCE_SNAP,
    "input_y": 0.4,
    "output_coords": [[x, y] for x, y in coords],
}
print("PASS: snap — near-miss vertices aligned to reference within tolerance")

In [ ]:
# ---------------------------------------------------------------------------
# Benchmark — throughput for each cleaning function over a synthetic corpus.
# Single polygon input, repartition(N_PART, "id") for Serverless parallelism.
# ---------------------------------------------------------------------------
bench_wkts = [
    (i, "POLYGON((0 0,0 5,0.001 8,0 10,10 10,10 0,0 0))")
    for i in range(N_BENCH)
]
df_bench = (
    spark.createDataFrame(bench_wkts, "id INT, wkt STRING")
    .repartition(N_PART, "id")
    .cache()
)
df_bench.count()  # materialize cache
print(f"Benchmark corpus: {N_BENCH} rows, {N_PART} partitions (polygon, cached)")

bench_results = {}
for fn_name, sql_expr in [
    ("simplifypreservetopology", "gbx_st_simplifypreservetopology(wkt, 1.0)"),
    ("removerepeatedpoints", "gbx_st_removerepeatedpoints(wkt)"),
    ("reduceprecision", "gbx_st_reduceprecision(wkt, 1.0)"),
    ("node", "gbx_st_node(wkt)"),
    ("snap", "gbx_st_snap(wkt, 'LINESTRING(0 0,10 0)', 0.5)"),
]:
    t0 = time.time()
    n = df_bench.selectExpr(f"{sql_expr} AS out").count()
    elapsed = time.time() - t0
    rps = N_BENCH / elapsed if elapsed > 0 else 0.0
    bench_results[fn_name] = {
        "elapsed_sec": round(elapsed, 3),
        "rows_per_sec": round(rps, 1),
    }
    print(f"  {fn_name:30s}: {elapsed:.2f}s  ({rps:.0f} rows/s)")

results["benchmark"] = {
    "n_bench": N_BENCH,
    "n_partitions": N_PART,
    "per_function": bench_results,
    **{f"rows_per_sec_{k}": v["rows_per_sec"] for k, v in bench_results.items()},
}
print(f"\nBenchmark complete: {N_BENCH} rows x 5 functions")

In [ ]:
# ---------------------------------------------------------------------------
# Final: assemble results, assert overall pass, exit with structured output.
# ---------------------------------------------------------------------------
all_stage_pass = all(
    results["stages"].get(fn, {}).get("pass", False)
    for fn in [
        "simplifypreservetopology",
        "removerepeatedpoints",
        "reduceprecision",
        "node",
        "snap",
    ]
)
results["pass"] = all_stage_pass

print(json.dumps(results, indent=2))

if results["pass"]:
    print("\nPASS: all cleaning validation checks green.")
    for fn, stage in results["stages"].items():
        effect = stage.get("effect", "")
        print(f"  - {fn}: {effect}")
    bm = results.get("benchmark", {}).get("per_function", {})
    if bm:
        print("  benchmark throughput (rows/s):")
        for fn, v in bm.items():
            print(f"    {fn:30s}: {v['rows_per_sec']:.0f}")
else:
    print("\nFAILED: see results dict above for details.")

assert results["pass"], "Cleaning validation FAILED — see results dict for details."
dbutils.notebook.exit(json.dumps(results))